
Databricks Notebook: WC_BADGE_DETAILS_D ETL Process



###  STEP 1 :DEFINE WIDGETS AND PARAMETERS

In [0]:
%sql
-- Create widgets for runtime parameters
CREATE WIDGET TEXT ETL_JOB_TYPE DEFAULT 'EOD';
CREATE WIDGET TEXT DATASOURCE_NUM_ID DEFAULT '380';
CREATE WIDGET TEXT ETL_PROC_WID DEFAULT '1';

### Step 2: Get ETL Control Parameters

In [0]:
%sql
-- Get last extract time
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT etl_last_extract_time 
FROM workspace.PRXBI_DW.wc_etl_parameters 
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [0]:
%sql
-- Get current extract time
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT etl_current_extract_time 
FROM workspace.PRXBI_DW.wc_etl_parameters 
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [0]:
%sql
-- Get ROW_WID for ETL parameters
CREATE OR REPLACE TEMPORARY VIEW v_etl_row_wid AS
SELECT ROW_WID 
FROM workspace.PRXBI_DW.wc_etl_parameters 
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [0]:
%sql
-- Display parameters for verification
SELECT 
    'Last Extract Time' AS parameter, etl_last_extract_time AS value FROM v_etl_last_extract_time
UNION ALL
SELECT 
    'Current Extract Time' AS parameter, etl_current_extract_time AS value FROM v_etl_current_extract_time
    UNION ALL
SELECT 
    'ETL ROW_WID' AS parameter,
    CAST(ROW_WID AS STRING) AS value
FROM v_etl_row_wid;

parameter,value
Last Extract Time,2025-10-20T00:00:00.000Z
Current Extract Time,2026-01-07T00:00:00.000Z
ETL ROW_WID,+21937-01-01T00:00:00.000Z


### Step 3: Cleanup & Create Staging Table (C$)

In [0]:
%sql
-- Drop staging table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_badge_details_stg;

-- Create staging table with Delta format
CREATE TABLE workspace.PRXBI_DW.c_badge_details_stg (
    ID STRING,
    BADGELOCATION STRING,
    BADGETOKEN STRING,
    BADGEVERSION DECIMAL(5,0),
    CONTACTEMAIL STRING,
    CONTACTFIRSTNAME STRING,
    CONTACTJOBTITLE STRING,
    CONTACTLASTNAME STRING,
    CONTACTPERSONRXMASTERID STRING,
    CREATEDBYREGISTRATIONTYPE STRING,
    CREATEDBYTYPE STRING,
    CULTURE STRING,
    CUSTOMERTYPE STRING,
    EVENTEDITIONGBSCODE STRING,
    ISBADGEUPDATE STRING,
    -- CORRECTED COLUMN NAME BELOW
    MARKETINGPREFERENCESPROMPTREQUIRED STRING, 
    ORGANISATIONCITY STRING,
    ORGANISATIONCOUNTRYCODE STRING,
    ORGANISATIONDISPLAYNAME STRING,
    ORGANISATIONRXMASTERID STRING,
    ORGANISATIONSTATE STRING,
    PARTICIPATINGORGANISATIONID STRING,
    PRODUCTCODE STRING,
    QRCODECONTENT STRING,
    REGISTRATIONID STRING,
    STATUS DECIMAL(10,0),
    SUPPORTSTAFFCOMPANYADDRESS STRING,
    SUPPORTSTAFFCOMPANYNAME STRING,
    SUPPORTSTAFFMOBILEPHONE STRING,
    SUPPORTSTAFFREPORTSTO STRING,
    SUPPORTSTAFFSTANDS STRING,
    SUPPORTSTAFFUSERACCESS STRING,
    VERSIONNUMBER DECIMAL(10,0),
    MOBILEPHONE STRING,
    FIRSTSCANNEDDATE TIMESTAMP,
    LASTPRINTEDDATE TIMESTAMP,
    ACCESSVALIDITYMODIFIEDDATE TIMESTAMP,
    CREATEDDATE TIMESTAMP,
    COMPANYPRODUCTCODE STRING,
    PAYMENTSTATUS STRING,
    PHOTOKEY STRING,
    PHOTOSOURCE STRING,
    PHOTOSOURCETYPE STRING
)
USING DELTA;

### Step 4: Extract & Deduplicate Source Data

In [0]:
%sql
-- Extract incremental data with version deduplication
INSERT INTO workspace.PRXBI_DW.c_badge_details_stg
SELECT
    ID, BADGELOCATION, BADGETOKEN, BADGEVERSION, CONTACTEMAIL, 
    CONTACTFIRSTNAME, CONTACTJOBTITLE, CONTACTLASTNAME, CONTACTPERSONRXMASTERID,
    CREATEDBYREGISTRATIONTYPE, CREATEDBYTYPE, CULTURE, CUSTOMERTYPE,
    EVENTEDITIONGBSCODE, ISBADGEUPDATE, 
    -- CORRECTED COLUMN NAME BELOW
    MARKETINGPREFERENCESPROMPTREQUIRED,
    ORGANISATIONCITY, ORGANISATIONCOUNTRYCODE, ORGANISATIONDISPLAYNAME,
    ORGANISATIONRXMASTERID, ORGANISATIONSTATE, PARTICIPATINGORGANISATIONID,
    PRODUCTCODE, QRCODECONTENT, REGISTRATIONID, STATUS,
    SUPPORTSTAFFCOMPANYADDRESS, SUPPORTSTAFFCOMPANYNAME, SUPPORTSTAFFMOBILEPHONE,
    SUPPORTSTAFFREPORTSTO, SUPPORTSTAFFSTANDS, SUPPORTSTAFFUSERACCESS,
    VERSIONNUMBER, MOBILEPHONE, FIRSTSCANNEDDATE, LASTPRINTEDDATE,
    ACCESSVALIDITYMODIFIEDDATE, CREATEDDATE, COMPANYPRODUCTCODE,
    PAYMENTSTATUS, PHOTOKEY, PHOTOSOURCE, PHOTOSOURCETYPE
FROM (
    SELECT 
        TS.*,
        ROW_NUMBER() OVER(
            PARTITION BY TS.ID 
            ORDER BY TS.INT_INSERT_DATE DESC, TS.VERSIONNUMBER DESC
        ) AS RNK
    FROM workspace.prxbi_ts.wc_mercury_badge_ts TS
    WHERE TS.INT_INSERT_DATE > (SELECT etl_last_extract_time FROM v_etl_last_extract_time)
      AND TS.INT_INSERT_DATE <= (SELECT etl_current_extract_time FROM v_etl_current_extract_time)
)
WHERE RNK = 1;

num_affected_rows,num_inserted_rows
8753,8753


In [0]:

%sql
-- Validation
SELECT COUNT(*) AS staging_count FROM workspace.PRXBI_DW.c_badge_details_stg;

staging_count
76514


### Step 5: Create Integration/Flow Table (I$)

In [0]:
%sql
-- Drop integration table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_badge_details_flow;

-- Create integration table
CREATE TABLE workspace.PRXBI_DW.i_badge_details_flow (
    BADGE_ID STRING,
    BADGE_LOCATION STRING,
    BADGE_TOKEN STRING,
    BADGE_VERSION DECIMAL(5,0),
    CONTACT_EMAIL STRING,
    CONTACT_FIRST_NAME STRING,
    CONTACT_LAST_NAME STRING,
    CONTACT_JOB_TITLE STRING,
    CONTACT_PERSON_ID STRING,
    CREATION_REG_TYPE STRING,
    CREATION_TYPE STRING,
    CULTURE STRING,
    CUSTOMER_TYPE STRING,
    EVENT_EDITION_CODE STRING,
    BADGE_UPDATE_FLG STRING,
    MARKETING_PREF_PROMPT STRING,
    ORG_NAME STRING,
    ORG_CITY STRING,
    ORG_COUNTRY STRING,
    ORG_ID STRING,
    ORG_STATE STRING,
    PARTICIPATING_ORG_ID STRING,
    PRODUCT_CODE STRING,
    QR_CODE STRING,
    REGISTRATION_ID STRING,
    STATUS DECIMAL(10,0),
    STAFF_COMPANY_NAME STRING,
    STAFF_COMPANY_ADDR STRING,
    STAFF_PHONE_NUM STRING,
    STAFF_REPORTING STRING,
    STAFF_STANDS STRING,
    STAFF_USER_ACCESS STRING,
    VERSION_NUM DECIMAL(10,0),
    INTEGRATION_ID STRING,
    DATASOURCE_NUM_ID STRING,
    MOBILEPHONE STRING,
    FIRSTSCANNEDDATE TIMESTAMP,
    LASTPRINTEDDATE TIMESTAMP,
    FIRSTSCANNEDDATE_FLG STRING,
    LASTPRINTEDDATE_FLG STRING,
    ACCESSVALIDITYMODIFIEDDATE TIMESTAMP,
    CREATEDDATE TIMESTAMP,
    COMPANYPRODUCTCODE STRING,
    PAYMENTSTATUS STRING,
    PHOTOKEY STRING,
    PHOTOSOURCE STRING,
    PHOTOSOURCETYPE STRING,
    PACKAGE_NAME STRING,
    W_INSERT_DT TIMESTAMP,
    W_UPDATE_DT TIMESTAMP,
    IND_UPDATE STRING
)
USING DELTA;

### Step 6: Enrich & Detect Changes (I$ Load)

In [0]:
%sql
INSERT INTO workspace.PRXBI_DW.i_badge_details_flow
SELECT 
    S.ID AS BADGE_ID,
    S.BADGELOCATION AS BADGE_LOCATION,
    S.BADGETOKEN AS BADGE_TOKEN,
    S.BADGEVERSION AS BADGE_VERSION,
    S.CONTACTEMAIL AS CONTACT_EMAIL,
    S.CONTACTFIRSTNAME AS CONTACT_FIRST_NAME,
    S.CONTACTLASTNAME AS CONTACT_LAST_NAME,
    S.CONTACTJOBTITLE AS CONTACT_JOB_TITLE,
    S.CONTACTPERSONRXMASTERID AS CONTACT_PERSON_ID,
    S.CREATEDBYREGISTRATIONTYPE AS CREATION_REG_TYPE,
    S.CREATEDBYTYPE AS CREATION_TYPE,
    S.CULTURE AS CULTURE,
    S.CUSTOMERTYPE AS CUSTOMER_TYPE,
    S.EVENTEDITIONGBSCODE AS EVENT_EDITION_CODE,
    S.ISBADGEUPDATE AS BADGE_UPDATE_FLG,
    S.MARKETINGPREFERENCESPROMPTREQUIRED AS MARKETING_PREF_PROMPT,
    S.ORGANISATIONDISPLAYNAME AS ORG_NAME,
    S.ORGANISATIONCITY AS ORG_CITY,
    S.ORGANISATIONCOUNTRYCODE AS ORG_COUNTRY,
    S.ORGANISATIONRXMASTERID AS ORG_ID,
    S.ORGANISATIONSTATE AS ORG_STATE,
    S.PARTICIPATINGORGANISATIONID AS PARTICIPATING_ORG_ID,
    S.PRODUCTCODE AS PRODUCT_CODE,
    S.QRCODECONTENT AS QR_CODE,
    S.REGISTRATIONID AS REGISTRATION_ID,
    S.STATUS AS STATUS,
    S.SUPPORTSTAFFCOMPANYNAME AS STAFF_COMPANY_NAME,
    S.SUPPORTSTAFFCOMPANYADDRESS AS STAFF_COMPANY_ADDR,
    S.SUPPORTSTAFFMOBILEPHONE AS STAFF_PHONE_NUM,
    S.SUPPORTSTAFFREPORTSTO AS STAFF_REPORTING,
    S.SUPPORTSTAFFSTANDS AS STAFF_STANDS,
    S.SUPPORTSTAFFUSERACCESS AS STAFF_USER_ACCESS,
    S.VERSIONNUMBER AS VERSION_NUM,
    S.ID AS INTEGRATION_ID,
    '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
    S.MOBILEPHONE,
    S.FIRSTSCANNEDDATE,
    S.LASTPRINTEDDATE,
    CASE WHEN S.FIRSTSCANNEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS FIRSTSCANNEDDATE_FLG,
    CASE WHEN S.LASTPRINTEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS LASTPRINTEDDATE_FLG,
    S.ACCESSVALIDITYMODIFIEDDATE,
    S.CREATEDDATE,
    S.COMPANYPRODUCTCODE,
    S.PAYMENTSTATUS,
    S.PHOTOKEY,
    S.PHOTOSOURCE,
    S.PHOTOSOURCETYPE,
    PROD.NAME AS PACKAGE_NAME,
    CURRENT_TIMESTAMP() AS W_INSERT_DT,
    CURRENT_TIMESTAMP() AS W_UPDATE_DT,
    'I' AS IND_UPDATE
FROM workspace.PRXBI_DW.c_badge_details_stg S
-- Join with Product Source (Deduplicated logic as per ODI)
LEFT JOIN (
    SELECT SKU, NAME, 
           ROW_NUMBER() OVER (PARTITION BY SKU ORDER BY ID DESC) as rn
    FROM workspace.prxbi_dw.wc_badge_product_d
) PROD ON S.PRODUCTCODE = PROD.SKU AND PROD.rn = 1
-- Change Detection (NOT EXISTS)
WHERE NOT EXISTS (
    SELECT 1 
    FROM workspace.PRXBI_DW.wc_badge_details_d T
    WHERE T.INTEGRATION_ID = S.ID
      AND T.DATASOURCE_NUM_ID = '${DATASOURCE_NUM_ID}'
      -- Check for any column changes (NULL safe comparison)
      AND (
          T.BADGE_LOCATION <=> S.BADGELOCATION AND
          T.CONTACT_EMAIL <=> S.CONTACTEMAIL AND
          T.STATUS <=> S.STATUS AND
          T.PRODUCT_CODE <=> S.PRODUCTCODE AND
          T.PACKAGE_NAME <=> PROD.NAME
          -- Add all other 40+ columns here for full CDC accuracy
      )
);

num_affected_rows,num_inserted_rows
76514,76514


### Step 7: Flag Updates

In [0]:
%sql
UPDATE workspace.PRXBI_DW.i_badge_details_flow F
SET IND_UPDATE = 'U'
WHERE EXISTS (
    SELECT 1 
    FROM workspace.PRXBI_DW.wc_badge_details_d T
    WHERE T.INTEGRATION_ID = F.INTEGRATION_ID
      AND T.DATASOURCE_NUM_ID = F.DATASOURCE_NUM_ID
);

-- Check breakdown
SELECT IND_UPDATE, COUNT(*) FROM workspace.PRXBI_DW.i_badge_details_flow GROUP BY IND_UPDATE;

IND_UPDATE,COUNT(*)
I,76514


### Step 8: MERGE (Upsert) to Target

In [0]:
%sql
MERGE INTO workspace.PRXBI_DW.wc_badge_details_d AS T
USING workspace.PRXBI_DW.i_badge_details_flow AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID 
   AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID

-- Update Existing Records
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.BADGE_ID = S.BADGE_ID,
    T.BADGE_LOCATION = S.BADGE_LOCATION,
    T.BADGE_TOKEN = S.BADGE_TOKEN,
    T.BADGE_VERSION = S.BADGE_VERSION,
    T.CONTACT_EMAIL = S.CONTACT_EMAIL,
    T.CONTACT_FIRST_NAME = S.CONTACT_FIRST_NAME,
    T.CONTACT_LAST_NAME = S.CONTACT_LAST_NAME,
    T.CONTACT_JOB_TITLE = S.CONTACT_JOB_TITLE,
    T.CONTACT_PERSON_ID = S.CONTACT_PERSON_ID,
    T.CREATION_REG_TYPE = S.CREATION_REG_TYPE,
    T.CREATION_TYPE = S.CREATION_TYPE,
    T.CULTURE = S.CULTURE,
    T.CUSTOMER_TYPE = S.CUSTOMER_TYPE,
    T.EVENT_EDITION_CODE = S.EVENT_EDITION_CODE,
    T.BADGE_UPDATE_FLG = S.BADGE_UPDATE_FLG,
    T.MARKETING_PREF_PROMPT = S.MARKETING_PREF_PROMPT,
    T.ORG_NAME = S.ORG_NAME,
    T.ORG_CITY = S.ORG_CITY,
    T.ORG_COUNTRY = S.ORG_COUNTRY,
    T.ORG_ID = S.ORG_ID,
    T.ORG_STATE = S.ORG_STATE,
    T.PARTICIPATING_ORG_ID = S.PARTICIPATING_ORG_ID,
    T.PRODUCT_CODE = S.PRODUCT_CODE,
    T.QR_CODE = S.QR_CODE,
    T.REGISTRATION_ID = S.REGISTRATION_ID,
    T.STATUS = S.STATUS,
    T.STAFF_COMPANY_NAME = S.STAFF_COMPANY_NAME,
    T.STAFF_COMPANY_ADDR = S.STAFF_COMPANY_ADDR,
    T.STAFF_PHONE_NUM = S.STAFF_PHONE_NUM,
    T.STAFF_REPORTING = S.STAFF_REPORTING,
    T.STAFF_STANDS = S.STAFF_STANDS,
    T.STAFF_USER_ACCESS = S.STAFF_USER_ACCESS,
    T.VERSION_NUM = S.VERSION_NUM,
    T.MOBILEPHONE = S.MOBILEPHONE,
    T.FIRSTSCANNEDDATE = S.FIRSTSCANNEDDATE,
    T.LASTPRINTEDDATE = S.LASTPRINTEDDATE,
    T.FIRSTSCANNEDDATE_FLG = S.FIRSTSCANNEDDATE_FLG,
    T.LASTPRINTEDDATE_FLG = S.LASTPRINTEDDATE_FLG,
    T.ACCESSVALIDITYMODIFIEDDATE = S.ACCESSVALIDITYMODIFIEDDATE,
    T.CREATEDDATE = S.CREATEDDATE,
    T.COMPANYPRODUCTCODE = S.COMPANYPRODUCTCODE,
    T.PAYMENTSTATUS = S.PAYMENTSTATUS,
    T.PHOTOKEY = S.PHOTOKEY,
    T.PHOTOSOURCE = S.PHOTOSOURCE,
    T.PHOTOSOURCETYPE = S.PHOTOSOURCETYPE,
    T.PACKAGE_NAME = S.PACKAGE_NAME,
    T.W_UPDATE_DT = CURRENT_TIMESTAMP()

-- Insert New Records
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
    BADGE_ID, BADGE_LOCATION, BADGE_TOKEN, BADGE_VERSION, CONTACT_EMAIL,
    CONTACT_FIRST_NAME, CONTACT_LAST_NAME, CONTACT_JOB_TITLE, CONTACT_PERSON_ID,
    CREATION_REG_TYPE, CREATION_TYPE, CULTURE, CUSTOMER_TYPE, EVENT_EDITION_CODE,
    BADGE_UPDATE_FLG, MARKETING_PREF_PROMPT, ORG_NAME, ORG_CITY, ORG_COUNTRY,
    ORG_ID, ORG_STATE, PARTICIPATING_ORG_ID, PRODUCT_CODE, QR_CODE, REGISTRATION_ID,
    STATUS, STAFF_COMPANY_NAME, STAFF_COMPANY_ADDR, STAFF_PHONE_NUM, STAFF_REPORTING,
    STAFF_STANDS, STAFF_USER_ACCESS, VERSION_NUM, INTEGRATION_ID, DATASOURCE_NUM_ID,
    MOBILEPHONE, FIRSTSCANNEDDATE, LASTPRINTEDDATE, FIRSTSCANNEDDATE_FLG,
    LASTPRINTEDDATE_FLG, ACCESSVALIDITYMODIFIEDDATE, CREATEDDATE, COMPANYPRODUCTCODE,
    PAYMENTSTATUS, PHOTOKEY, PHOTOSOURCE, PHOTOSOURCETYPE, PACKAGE_NAME,
    W_INSERT_DT, W_UPDATE_DT
) VALUES (
    S.BADGE_ID, S.BADGE_LOCATION, S.BADGE_TOKEN, S.BADGE_VERSION, S.CONTACT_EMAIL,
    S.CONTACT_FIRST_NAME, S.CONTACT_LAST_NAME, S.CONTACT_JOB_TITLE, S.CONTACT_PERSON_ID,
    S.CREATION_REG_TYPE, S.CREATION_TYPE, S.CULTURE, S.CUSTOMER_TYPE, S.EVENT_EDITION_CODE,
    S.BADGE_UPDATE_FLG, S.MARKETING_PREF_PROMPT, S.ORG_NAME, S.ORG_CITY, S.ORG_COUNTRY,
    S.ORG_ID, S.ORG_STATE, S.PARTICIPATING_ORG_ID, S.PRODUCT_CODE, S.QR_CODE, S.REGISTRATION_ID,
    S.STATUS, S.STAFF_COMPANY_NAME, S.STAFF_COMPANY_ADDR, S.STAFF_PHONE_NUM, S.STAFF_REPORTING,
    S.STAFF_STANDS, S.STAFF_USER_ACCESS, S.VERSION_NUM, S.INTEGRATION_ID, S.DATASOURCE_NUM_ID,
    S.MOBILEPHONE, S.FIRSTSCANNEDDATE, S.LASTPRINTEDDATE, S.FIRSTSCANNEDDATE_FLG,
    S.LASTPRINTEDDATE_FLG, S.ACCESSVALIDITYMODIFIEDDATE, S.CREATEDDATE, S.COMPANYPRODUCTCODE,
    S.PAYMENTSTATUS, S.PHOTOKEY, S.PHOTOSOURCE, S.PHOTOSOURCETYPE, S.PACKAGE_NAME,
    CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
76514,0,0,76514


### Step 9: Optimization & Cleanup

In [0]:
%sql
-- Optimize Target Table
--OPTIMIZE workspace.badge_schema_k.wc_badge_details_d
--ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);

-- Drop Temporary Tables
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_badge_details_flow;
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_badge_details_stg;

### Step 10: Final Validation

In [0]:
%sql
SELECT 
    'WC_BADGE_DETAILS_D' AS TableName,
    COUNT(*) AS TotalRecords,
    MAX(W_UPDATE_DT) AS LastUpdate
FROM workspace.PRXBI_DW.wc_badge_details_d
WHERE DATASOURCE_NUM_ID = '${DATASOURCE_NUM_ID}';

TableName,TotalRecords,LastUpdate
WC_BADGE_DETAILS_D,76514,2026-01-09
